# Domain 1: Agentic Architecture & Orchestration
## Task Statement 1.5: Apply Agent SDK Hooks for Tool Call Interception and Data Normalization
**Certification Weight**: 27% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to use Agent SDK lifecycle hooks (`PreToolUse` and `PostToolUse`) to implement deterministic guardrails, intercept sensitive tool invocations, and normalize heterogeneous API responses before model consumption.

---

## 🏗️ Architecture: Lifecycle Hooks Pipeline

```mermaid
flowchart LR
    ModelTurn["Claude Tool Call<br>(tool_use)"] --> PreHook["PreToolUse Hook<br>(Deterministic Policy Guard)"]
    
    PreHook --> PolicyCheck{"Policy Valid?<br>e.g. Amount <= $500"}
    
    PolicyCheck -->|"Violated"| BlockExec["Short-Circuit Tool Execution<br>Return PolicyBlocked Error"]
    PolicyCheck -->|"Approved"| ToolExec["Execute External Tool / API"]
    
    ToolExec --> PostHook["PostToolUse Hook<br>(Normalization & Trimming)"]
    
    PostHook --> CleanFormat["1. Normalize dates to ISO-8601<br>2. Map HTTP codes to Enums<br>3. Strip verbose debug headers"]
    
    CleanFormat --> ModelInput["Append Sanitized tool_result<br>to Claude History"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Hook Type | Primary Exam Purpose | Exam Trap to Reject |
|---|---|---|
| **`PreToolUse` Hook** | **Policy Enforcement & Safety Gating**: Inspect arguments before execution; abort or redirect unauthorized actions (e.g. budget cap, rate limit, permission check). | **TRAP**: Letting tools execute and then relying on post-execution checks to "undo" irreversible destructive changes. |
| **`PostToolUse` Hook** | **Normalization & Context Trimming**: Standardize data types, strip verbose payload fields, convert Unix epochs to ISO 8601, and map HTTP codes to semantic enums. | **TRAP**: Passing raw, 20-page vendor JSON responses with internal stack traces directly into model context. |
| **Enforcement Guarantee** | **100% Deterministic Code Execution**: Hooks run in the host application runtime, outside the LLM reasoning process. | **TRAP**: Asking the LLM in the system prompt to "please format raw tool outputs and obey spend caps". |

---

## 💡 Architectural Decision Matrix

| Requirement | Recommended Hook | Implementation Detail |
|---|---|---|
| **Enforce spend/rate limits** | `PreToolUse` | Intercept arguments, compare against database state or policy table, raise structured error if exceeded. |
| **Sanitize PII / Redact Tokens** | `PostToolUse` | Regex-strip credit cards, API keys, or JWT tokens before payload is stored in conversation context. |
| **Token Conservation** | `PostToolUse` | Strip redundant schema boilerplate, pagination cursors, and keep only requested query columns. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.5 Agent SDK Hooks.")

### 🎣 Step 2: PreToolUse & PostToolUse Hook Declarations
- `PreToolUse` checks financial boundaries (transactions > $500 require manager approval).
- `PostToolUse` normalizes epoch timestamps to ISO-8601 strings and strips internal database IDs.

In [ ]:
# Step 2: Hook Implementations
def compliance_interception_hook(tool_name: str, tool_input: Dict[str, Any]) -> Tuple[bool, Optional[Dict[str, Any]]]:
    """PreToolUse hook: Blocks payouts over $500 and enforces supervisor escalation."""
    if tool_name == "process_payout":
        amount = tool_input.get("amount", 0.0)
        if amount > 500.0:
            return False, {
                "blocked_by_hook": True,
                "reason": f"Payout amount ${amount:.2f} exceeds automated policy threshold ($500.00).",
                "redirect_action": "escalate_to_supervisor",
            }
    return True, None

def post_tool_use_normalization_hook(tool_name: str, raw_output: Dict[str, Any]) -> Dict[str, Any]:
    """PostToolUse hook: Normalizes timestamps, sanitizes internal fields, converts status codes."""
    normalized = dict(raw_output)
    # Convert unix epoch to ISO-8601
    if "epoch_timestamp" in normalized:
        ts = normalized.pop("epoch_timestamp")
        normalized["iso_timestamp"] = datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    # Normalize HTTP status codes to clean enums
    if "raw_status_code" in normalized:
        code = normalized.pop("raw_status_code")
        normalized["status"] = "active" if code == 200 else "inactive"
    # Redact sensitive internal tokens
    normalized.pop("internal_db_row_id", None)
    return normalized

print(" Hooks defined for PreToolUse and PostToolUse.")

### ⚙️ Step 3: Hook Execution Middleware
The wrapper integrates hooks around tool execution.

In [ ]:
# Step 3: Hook Middleware Wrapper
def execute_with_hooks(
    tool_name: str,
    tool_input: Dict[str, Any],
    tool_executor: Callable[[str, Dict[str, Any]], Dict[str, Any]],
) -> Dict[str, Any]:
    """Wraps tool execution with PreToolUse interception and PostToolUse normalization."""
    # 1. PreToolUse Hook
    allowed, intercept_result = compliance_interception_hook(tool_name, tool_input)
    if not allowed:
        return {"isError": True, "hook_interception": intercept_result}

    # 2. Execute Underlying Tool
    raw_output = tool_executor(tool_name, tool_input)

    # 3. PostToolUse Hook
    return post_tool_use_normalization_hook(tool_name, raw_output)

def sample_tool_executor(tool_name: str, inputs: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name == "check_account_status":
        return {
            "account_id": inputs["account_id"],
            "epoch_timestamp": 1728216000,
            "raw_status_code": 200,
            "internal_db_row_id": 994821,
        }
    elif tool_name == "process_payout":
        return {"status": "paid", "amount": inputs["amount"]}
    return {"error": "unknown"}

print(" Hook middleware initialized.")

### 🧪 Step 4: Simulation Walkthrough
1. Payout of $250 -> Permitted.
2. Payout of $800 -> Intercepted and blocked by PreToolUse hook.
3. Account query -> Output normalized by PostToolUse hook.

In [ ]:
# Step 4: Simulation Execution
# Case A: Below threshold payout ($250)
res_a = execute_with_hooks("process_payout", {"amount": 250.0}, sample_tool_executor)
print("Below Threshold Result:", res_a)

# Case B: Above threshold payout ($800) -> Intercepted
res_b = execute_with_hooks("process_payout", {"amount": 800.0}, sample_tool_executor)
print("\nAbove Threshold Result:", res_b)

# Case C: Account check -> Normalization
res_c = execute_with_hooks("check_account_status", {"account_id": "ACC-100"}, sample_tool_executor)
print("\nNormalized Account Output:", json.dumps(res_c, indent=2))

### 🔬 Step 5: Verification Assertions
We assert that:
1. Transactions over $500 are deterministically blocked.
2. Internal tokens and raw HTTP codes are cleanly sanitized before reaching Claude.

In [ ]:
# Step 5: Verification Assertions
assert res_a["status"] == "paid"
assert res_b["isError"] is True
assert res_b["hook_interception"]["blocked_by_hook"] is True

# Normalization assertions
assert "iso_timestamp" in res_c
assert "epoch_timestamp" not in res_c
assert "internal_db_row_id" not in res_c
assert res_c["status"] == "active"

print(" All verification assertions passed! SDK hooks validated.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use `PreToolUse` hooks for hard financial limits and security boundaries.
- **Rule 2**: Use `PostToolUse` hooks to normalize heterogeneous data types (timestamps, enums) and scrub sensitive internal fields.
- **Rule 3**: Never rely on Claude's prompt adherence when regulatory or financial thresholds must be strictly enforced.